# Regression suite — guard every snapshot

Two complementary checks keep the eval inventory honest:

1. **Cross-ref diff** (`.github/scripts/diff_eval_snapshots.py`) — runs in CI on every PR, compares head vs base branch with a 5% tolerance.
2. **Per-leaf baseline** (this leaf) — drop a `baseline.json` next to your leaf's `eval-snapshot.json` to assert a *hard* threshold that survives even a forced overwrite.

Together they catch both gradual drift and explicit overrides.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
sys.path.insert(0, str(ROOT))
from regression import walk_snapshots, flatten, direction
print('repo root:', ROOT)

## Walk every snapshot in the repo

In [ ]:
rows = []
for path, snap in walk_snapshots(ROOT):
    flat = {}
    flatten('', snap.get('metrics') or {}, flat)
    rows.append((snap.get('technique', path.parent.name), len(flat), path.relative_to(ROOT)))
print(f'{"technique":<28}{"#metrics":>10}  path')
for tech, n, p in rows[:15]:
    print(f'{tech:<28}{n:>10}  {p}')
print(f'... {len(rows)} snapshots total')

## Classify each metric — higher vs lower vs neutral

In [ ]:
n_higher = n_lower = n_neutral = 0
for path, snap in walk_snapshots(ROOT):
    flat = {}
    flatten('', snap.get('metrics') or {}, flat)
    for k in flat:
        d = direction(k)
        n_higher += d == 1
        n_lower  += d == -1
        n_neutral += d is None
print(f'higher-is-better: {n_higher}')
print(f'lower-is-better : {n_lower}')
print(f'neutral         : {n_neutral}  (config knobs, counts, IDs)')

## Opt in to a per-leaf baseline

Drop a `baseline.json` next to this leaf's `eval-snapshot.json` (or any other leaf's). Format:

```json
{
  "naive-rag": {"metrics.context_recall": 0.80},
  "hybrid-search": {"metrics.per_retriever.hybrid_rrf.recall@3": 0.95}
}
```

Then run `uv run pytest 05-evals-and-observability/05-regression-suite/tests` — the test will fail when any baselined metric regresses by > 5%.